# 04. Merging, Sorting and Rearranging Lists

Operations on one or two lists: merge, sort, partition, remove duplicates, add numbers, intersection.

**Learning order:** MERGE TWO → SORT LIST → DUPLICATES → PARTITION → ODD EVEN → INTERSECTION → ADD NUMBERS → MERGE K → INTERVIEW THINKING

## 1. Merge two sorted lists

Same merge as in merge sort. Use a **dummy** head and a `tail` pointer. Take the smaller front node, attach it, move on. Attach the leftover list at the end (no copying).

Real life: merging two sorted queues of tickets into one line.

In [1]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def build(vals):                      # list -> linked list
    dummy = ListNode()
    cur = dummy
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
    return dummy.next

def to_list(head):                    # linked list -> list (for printing)
    out = []
    while head:
        out.append(head.val)
        head = head.next
    return out

def merge_two(a, b):
    dummy = tail = ListNode()
    while a and b:
        if a.val <= b.val:
            tail.next, a = a, a.next
        else:
            tail.next, b = b, b.next
        tail = tail.next
    tail.next = a or b
    return dummy.next

print(to_list(merge_two(build([1, 2, 4]), build([1, 3, 4]))))   # [1, 1, 2, 3, 4, 4]

[1, 1, 2, 3, 4, 4]


## 2. Sort a list: merge sort

Arrays are sorted with quick sort, but for linked lists **merge sort** is the natural fit (no random access needed, O(1) extra space apart from recursion).

1. Find the middle with slow/fast and **cut** the list.
2. Sort both halves recursively.
3. Merge.

O(n log n) time, O(log n) stack.

In [2]:
def sort_list(head):
    if not head or not head.next:
        return head
    slow, fast = head, head.next
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    second, slow.next = slow.next, None               # cut
    return merge_two(sort_list(head), sort_list(second))

print(to_list(sort_list(build([4, 2, 1, 3]))))   # [1, 2, 3, 4]

[1, 2, 3, 4]


## 3. Remove duplicates from a sorted list

- **Keep one copy:** if `cur.val == cur.next.val`, skip `cur.next`.
- **Remove all values that repeat:** use a dummy and a `prev`. When a run of equal values is found, skip the whole run.

In [3]:
def delete_duplicates_keep_one(head):
    cur = head
    while cur and cur.next:
        if cur.val == cur.next.val:
            cur.next = cur.next.next
        else:
            cur = cur.next
    return head

def delete_duplicates_all(head):
    dummy = ListNode(0, head)
    prev = dummy
    while head:
        if head.next and head.val == head.next.val:
            while head.next and head.val == head.next.val:
                head = head.next
            prev.next = head.next                      # skip the whole run
        else:
            prev = prev.next
        head = head.next
    return dummy.next

print(to_list(delete_duplicates_keep_one(build([1, 1, 2, 3, 3]))))      # [1, 2, 3]
print(to_list(delete_duplicates_all(build([1, 2, 3, 3, 4, 4, 5]))))     # [1, 2, 5]

[1, 2, 3]
[1, 2, 5]


## 4. Partition list around `x`

Make two lists: nodes `< x` and nodes `>= x`, keeping the original order, then join them. Two dummy heads.

In [4]:
def partition(head, x):
    small_dummy, big_dummy = ListNode(), ListNode()
    small, big = small_dummy, big_dummy
    while head:
        if head.val < x:
            small.next = head
            small = small.next
        else:
            big.next = head
            big = big.next
        head = head.next
    big.next = None                                    # avoid a cycle
    small.next = big_dummy.next
    return small_dummy.next

print(to_list(partition(build([1, 4, 3, 2, 5, 2]), 3)))   # [1, 2, 2, 4, 3, 5]

[1, 2, 2, 4, 3, 5]


## 5. Odd Even Linked List

Group all nodes at odd positions first, then even positions. Same two-chain idea, in place.

In [5]:
def odd_even_list(head):
    if not head:
        return head
    odd, even = head, head.next
    even_head = even
    while even and even.next:
        odd.next = even.next
        odd = odd.next
        even.next = odd.next
        even = even.next
    odd.next = even_head
    return head

print(to_list(odd_even_list(build([1, 2, 3, 4, 5]))))   # [1, 3, 5, 2, 4]

[1, 3, 5, 2, 4]


## 6. Intersection of two lists

Two lists may **share a tail** (same nodes, not only same values). Trick: walk `pa` through A then B, `pb` through B then A. Both walk `lenA + lenB` steps, so they meet at the intersection (or both reach `None` together).

In [6]:
def get_intersection(a, b):
    pa, pb = a, b
    while pa is not pb:
        pa = pa.next if pa else b
        pb = pb.next if pb else a
    return pa

shared = build([8, 4, 5])
a = build([4, 1]); a.next.next = shared
b = build([5, 6, 1]); b.next.next.next = shared
print(get_intersection(a, b).val)   # 8

8


## 7. Add Two Numbers (digits stored in reverse)

Add digit by digit with a `carry`, like school addition. Loop while either list or the carry remains.

In [7]:
def add_two_numbers(a, b):
    dummy = tail = ListNode()
    carry = 0
    while a or b or carry:
        total = carry + (a.val if a else 0) + (b.val if b else 0)
        carry, digit = divmod(total, 10)
        tail.next = ListNode(digit)
        tail = tail.next
        a = a.next if a else None
        b = b.next if b else None
    return dummy.next

print(to_list(add_two_numbers(build([2, 4, 3]), build([5, 6, 4]))))   # [7, 0, 8]  (342 + 465 = 807)

[7, 0, 8]


## 8. Merge K sorted lists (preview of heaps)

Put the head of each list in a **min-heap**. Pop the smallest, attach it, push its `next`. O(N log k) for N total nodes. The index `i` in the tuple avoids comparing nodes when values tie.

In [8]:
import heapq

def merge_k_lists(lists):
    heap = []
    for i, node in enumerate(lists):
        if node:
            heapq.heappush(heap, (node.val, i, node))
    dummy = tail = ListNode()
    while heap:
        _, i, node = heapq.heappop(heap)
        tail.next = node
        tail = tail.next
        if node.next:
            heapq.heappush(heap, (node.next.val, i, node.next))
    return dummy.next

ls = [build([1, 4, 5]), build([1, 3, 4]), build([2, 6])]
print(to_list(merge_k_lists(ls)))   # [1, 1, 2, 3, 4, 4, 5, 6]

[1, 1, 2, 3, 4, 4, 5, 6]


## 9. Interview thinking

**Signals:** "merge", "sort in O(n log n) / O(1) space", "rearrange", "common node", "numbers stored as lists".

**Say:** "I use a dummy head so the first node is not special, and a tail pointer to append."

**Common mistakes:** not cutting the list before sorting the halves, forgetting `big.next = None` in partition (cycle), comparing values instead of nodes in intersection, not handling the final carry, forgetting that leftover nodes can be attached directly.

## Quick Revision

- Dummy + tail = build a result list.
- Merge two sorted: attach smaller, leftover with `a or b`.
- Sort list: middle, **cut**, sort halves, merge.
- Partition/odd-even: two chains, then join.
- Intersection: `pa`/`pb` switch heads.
- K lists: min-heap of `(val, index, node)`.